# External Framework Validation — Anonymeter (Bank Marketing)

Runs in the isolated `tese_anonymeter` conda environment. Attack-based privacy
(singling out, linkability, inference) aligned with the Article 29 Working Party
criteria. Mirrors the Dataset 1 Anonymeter notebook, adapted to the Bank feature
space; the linkability auxiliary split is redefined for this dataset.

## 0. Configuration and paths

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

RANDOM_STATE = 123

# Folder with the CSV files, relative to this notebook (adjust if needed)
DATA_DIR = Path("data")

assert DATA_DIR.exists(), f"Folder not found: {DATA_DIR}"
print("Data folder:", DATA_DIR)

from anonymeter.evaluators import (
    SinglingOutEvaluator,
    LinkabilityEvaluator,
    InferenceEvaluator,
)

Data folder: data


## 1. Canonical column definitions (Bank Marketing)

In [2]:
TARGET = "y"

FEATURES = ['age','job','marital','education','default','housing','loan','contact',
            'month','day_of_week','campaign','previous','poutcome','emp.var.rate',
            'cons.price.idx','cons.conf.idx','euribor3m','nr.employed','was_contacted']

# Categorical columns: the 11 nominal categoricals (incl. the binary flag).
# Bank Marketing has NO ordinal columns, so there is a single, unambiguous
# categorical treatment (no Variant A/B, unlike the PAY_X columns of Dataset 1).
CAT_COLS = ['job','marital','education','default','housing','loan','contact',
            'month','day_of_week','poutcome','was_contacted']
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

assert len(FEATURES) == 19, f"FEATURES should have 19, got {len(FEATURES)}"
assert TARGET not in FEATURES, "y must not be in FEATURES"
assert len(CAT_COLS) == 11 and len(NUM_COLS) == 8
print("FEATURES:", len(FEATURES), "| CAT:", len(CAT_COLS), "| NUM:", len(NUM_COLS))

ALL_COLS = FEATURES + [TARGET]
print("Working schema:", len(ALL_COLS), "columns (19 features + target)")

FEATURES: 19 | CAT: 11 | NUM: 8
Working schema: 20 columns (19 features + target)


## 2. Helper: align a dataframe to the canonical schema

In [3]:
def align(df, name):
    d = df.copy()
    if TARGET not in d.columns:
        d[TARGET] = 1   # minority reference: all y == 1
    d = d[ALL_COLS]
    d[NUM_COLS] = d[NUM_COLS].astype("float64")
    d[TARGET] = d[TARGET].astype(int)
    assert list(d.columns) == ALL_COLS
    print(f"{name:<16} -> {d.shape}")
    return d

## 3. Load the real (ori) and synthetic (syn) datasets

In [4]:
ori       = align(pd.read_csv(DATA_DIR / "bank_real_minority_train.csv"),  "ori (real min)")
syn_ctgan = align(pd.read_csv(DATA_DIR / "bank_synthetic_ctgan_final.csv"), "syn ctgan")
syn_ctab  = align(pd.read_csv(DATA_DIR / "bank_synthetic_ctab_final.csv"),  "syn ctabgan+")

for d in (syn_ctgan, syn_ctab):
    assert list(d.columns) == list(ori.columns)
print("\nColumns aligned across ori and syn.")

ori (real min)   -> (3248, 20)
syn ctgan        -> (22335, 20)
syn ctabgan+     -> (22335, 20)

Columns aligned across ori and syn.


## 4. Reconstruct the canonical split → minority control

In [5]:
df_raw = pd.read_csv(DATA_DIR / "bank-additional-full.csv", sep=';')

df_c = df_raw.drop(columns=['duration'])
df_c['was_contacted'] = (df_c['pdays'] != 999).astype(int)
df_c = df_c.drop(columns=['pdays'])
df_c[TARGET] = (df_c['y'] == 'yes').astype(int)
df_c['default'] = df_c['default'].replace({'yes': 'unknown'})

X = df_c.drop(columns=TARGET)
y = df_c[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

# verify reproduction against the stored training partition
real_full = pd.read_csv(DATA_DIR / "bank_real_train_full.csv")
train_recon = X_train.copy()
train_recon[TARGET] = y_train
train_recon = train_recon[real_full.columns].reset_index(drop=True)
# Compare VALUES, not strict dtypes (int32 vs int64 store the same 0/1 data)
pd.testing.assert_frame_equal(
    train_recon, real_full.reset_index(drop=True),
    check_dtype=False,   # values are compared, not dtypes (int32 vs int64 store the same 0/1 data)
)
print("Split reproduced bit for bit (values). OK.")

# control = held-out test set, minority only, aligned to the canonical schema
holdout = X_test.copy()
holdout[TARGET] = y_test
control = align(holdout[holdout[TARGET] == 1], "control (min)")
print("Control (minority holdout):", control.shape)

Split reproduced bit for bit (values). OK.
control (min)    -> (1392, 20)
Control (minority holdout): (1392, 20)


## 5. Attack configuration
Linkability partitions the attributes into a demographic group and a
campaign/behaviour group (the Bank analogue of the demographic-vs-financial
split used in Dataset 1).

Note: the attacks draw their targets at random and `RANDOM_STATE` is not
passed to the evaluators, so repeated runs give slightly different risks,
within the reported confidence intervals.

In [6]:
N_ATTACKS = min(1500, len(control) - 1)

# Linkability: two disjoint auxiliary sets (demographic vs campaign/behaviour)
AUX_A = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
AUX_B = [c for c in ALL_COLS if c not in AUX_A]

assert set(AUX_A).isdisjoint(AUX_B)
assert set(AUX_A) | set(AUX_B) == set(ALL_COLS)
print("Linkability aux groups:", len(AUX_A), "vs", len(AUX_B), "columns")
print("N_ATTACKS =", N_ATTACKS, "| control size =", len(control))

Linkability aux groups: 7 vs 13 columns
N_ATTACKS = 1391 | control size = 1392


## 6. Singling Out risk

In [7]:
def singling_out(syn, name, mode, n_cols=None):
    kwargs = dict(ori=ori, syn=syn, control=control, n_attacks=N_ATTACKS)
    if mode == "multivariate":
        kwargs["n_cols"] = n_cols
    ev = SinglingOutEvaluator(**kwargs)
    ev.evaluate(mode=mode)
    r = ev.risk()
    print(f"{name:<12} [{mode:>12}]  risk = {r.value:.4f}  CI = "
          f"({r.ci[0]:.4f}, {r.ci[1]:.4f})")
    return {"model": name, "attack": f"singling_out_{mode}",
            "risk": r.value, "ci_low": r.ci[0], "ci_high": r.ci[1]}

so_rows = []
for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]:
    so_rows.append(singling_out(syn, name, "univariate"))
    so_rows.append(singling_out(syn, name, "multivariate", n_cols=3))

CTGAN        [  univariate]  risk = 0.0000  CI = (0.0000, 0.0020)
CTGAN        [multivariate]  risk = 0.0504  CI = (0.0350, 0.0658)
CTAB-GAN+    [  univariate]  risk = 0.0000  CI = (0.0000, 0.0020)
CTAB-GAN+    [multivariate]  risk = 0.0343  CI = (0.0195, 0.0491)


## 7. Linkability risk

In [8]:
def linkability(syn, name):
    ev = LinkabilityEvaluator(ori=ori, syn=syn, control=control,
                              n_attacks=N_ATTACKS,
                              aux_cols=(AUX_A, AUX_B),
                              n_neighbors=10)
    ev.evaluate()
    r = ev.risk()
    print(f"{name:<12} linkability  risk = {r.value:.4f}  CI = "
          f"({r.ci[0]:.4f}, {r.ci[1]:.4f})")
    return {"model": name, "attack": "linkability",
            "risk": r.value, "ci_low": r.ci[0], "ci_high": r.ci[1]}

link_rows = [linkability(syn, name)
             for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]]

CTGAN        linkability  risk = 0.0007  CI = (0.0000, 0.0065)
CTAB-GAN+    linkability  risk = 0.0000  CI = (0.0000, 0.0028)


## 8. Inference risk (per attribute)

In [9]:
def inference_all(syn, name):
    rows = []
    for secret in ALL_COLS:
        aux = [c for c in ALL_COLS if c != secret]
        ev = InferenceEvaluator(ori=ori, syn=syn, control=control,
                                n_attacks=N_ATTACKS,
                                aux_cols=aux, secret=secret)
        ev.evaluate()
        r = ev.risk()
        rows.append({"model": name, "attack": "inference", "secret": secret,
                     "risk": r.value, "ci_low": r.ci[0], "ci_high": r.ci[1]})
    return rows

inf_rows = []
for name, syn in [("CTGAN", syn_ctgan), ("CTAB-GAN+", syn_ctab)]:
    print(f"\nInference attacks for {name} (one per column)...")
    rows = inference_all(syn, name)
    inf_rows.extend(rows)
    top = (pd.DataFrame(rows).sort_values("risk", ascending=False)
           .head(5)[["secret", "risk"]])
    print(top.to_string(index=False))


Inference attacks for CTGAN (one per column)...
       secret     risk
      contact 0.119295
      default 0.076951
     previous 0.047179
was_contacted 0.036049
     campaign 0.027918

Inference attacks for CTAB-GAN+ (one per column)...
       secret     risk
was_contacted 0.089528
          job 0.012734
     previous 0.007408
      housing 0.005587
    education 0.003677


## 9. Consolidate and save

In [10]:
summary_df = pd.DataFrame(so_rows + link_rows)
inference_df = pd.DataFrame(inf_rows)

summary_df.to_csv(DATA_DIR / "bank_anonymeter_summary_results.csv", index=False)
inference_df.to_csv(DATA_DIR / "bank_anonymeter_inference_results.csv", index=False)
print("Saved bank_anonymeter_*.csv")

print("\n=== SINGLING OUT + LINKABILITY ===")
print(summary_df.pivot(index="model", columns="attack", values="risk").round(4))
print("\n=== INFERENCE (mean risk per model) ===")
print(inference_df.groupby("model")["risk"].mean().round(4))

Saved bank_anonymeter_*.csv

=== SINGLING OUT + LINKABILITY ===
attack     linkability  singling_out_multivariate  singling_out_univariate
model                                                                     
CTAB-GAN+       0.0000                     0.0343                      0.0
CTGAN           0.0007                     0.0504                      0.0

=== INFERENCE (mean risk per model) ===
model
CTAB-GAN+    0.0059
CTGAN        0.0176
Name: risk, dtype: float64


In [12]:
# Show the CIs of the top inference attributes to check which are significant
for model in ["CTGAN", "CTAB-GAN+"]:
    print(f"\n=== {model} — inference (top 8 by risk, with CIs) ===")
    sub = (inference_df[inference_df["model"] == model]
           .sort_values("risk", ascending=False)
           .head(8))
    for _, r in sub.iterrows():
        sig = "SIGNIFICANT" if r["ci_low"] > 0 else "not sig. (CI includes 0)"
        print(f"  {r['secret']:<16} risk={r['risk']:.4f}  "
              f"CI=[{r['ci_low']:.4f}, {r['ci_high']:.4f}]  {sig}")


=== CTGAN — inference (top 8 by risk, with CIs) ===
  contact          risk=0.1193  CI=[0.0000, 0.2425]  not sig. (CI includes 0)
  default          risk=0.0770  CI=[0.0000, 0.2385]  not sig. (CI includes 0)
  previous         risk=0.0472  CI=[0.0000, 0.1561]  not sig. (CI includes 0)
  was_contacted    risk=0.0360  CI=[0.0000, 0.2021]  not sig. (CI includes 0)
  campaign         risk=0.0279  CI=[0.0000, 0.0741]  not sig. (CI includes 0)
  marital          risk=0.0207  CI=[0.0000, 0.0916]  not sig. (CI includes 0)
  emp.var.rate     risk=0.0159  CI=[0.0000, 0.0349]  not sig. (CI includes 0)
  age              risk=0.0044  CI=[0.0000, 0.0152]  not sig. (CI includes 0)

=== CTAB-GAN+ — inference (top 8 by risk, with CIs) ===
  was_contacted    risk=0.0895  CI=[0.0000, 0.2162]  not sig. (CI includes 0)
  job              risk=0.0127  CI=[0.0000, 0.0448]  not sig. (CI includes 0)
  previous         risk=0.0074  CI=[0.0000, 0.0838]  not sig. (CI includes 0)
  housing          risk=0.0056  

---